In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ["intel-isl_multiobjectiveoptimization", "sjmoran_deeplpf", "pangeo-forge_pangeo-forge", "lvrcek_gnnome-assembly",
"josiahcarlson_redis-in-action","nmoehrle_mvs-texturing","climateglobalchange_tempestremap","miguelinux314_experiment-notebook",
"rllab-snu_unsupervised-3d-reconstruction","coryshain_cdr"]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,034c7061f35f862db0220636e9a671efd3ba806c,intel-isl_multiobjectiveoptimization


In [4]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 38%|███▊      | 425/1108 [07:11<11:33,  1.01s/it]

Got Errors {'394b8069db94e2d2052c898db695977f9f89154d': 'Key 394b8069db94e2d2052c898db695977f9f89154d not found in /da5_fast/All.sha1c/commit_57.tch'}


 42%|████▏     | 463/1108 [07:50<10:54,  1.01s/it]

Got Errors {'38677165df4ab72c105b4d9a052d78e47762683f': 'Key 38677165df4ab72c105b4d9a052d78e47762683f not found in /da5_fast/All.sha1c/commit_56.tch'}


 94%|█████████▎| 1038/1108 [17:39<01:13,  1.04s/it]

Got Errors {'76294a4bc9e2c0d822f54d9ecdc8010610d07b00': 'Key 76294a4bc9e2c0d822f54d9ecdc8010610d07b00 not found in /da5_fast/All.sha1c/commit_118.tch'}


100%|██████████| 1108/1108 [18:52<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,034c7061f35f862db0220636e9a671efd3ba806c,594256cdbfde3edcce1e08ec922bc7607f18f717,[5d8e8343c56cf3184081f71cc4a661af64e7cf7e],CYH <1131637423@qq.com>,1617625010,+0800,CYH <1131637423@qq.com>,1617625010,+0800,Add data parallelism support\n,034c7061f35f862db0220636e9a671efd3ba806c,intel-isl_multiobjectiveoptimization
1,0f9472c8b7317b8b955dc88c281fa9c4276a493b,6b8b04693d2c2b8b5b46e35905776af125053bab,"[5d8e8343c56cf3184081f71cc4a661af64e7cf7e, f29...",Shahnawaz Alam <shahnawaz.alam.dev@gmail.com>,1589948368,+0200,GitHub <noreply@github.com>,1589948368,+0200,Merge f295df07782c6d3e19b78156bd3b12dc8eec2692...,0f9472c8b7317b8b955dc88c281fa9c4276a493b,intel-isl_multiobjectiveoptimization


In [14]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '034c7061f35f862db0220636e9a671efd3ba806c', 'tree': '594256cdbfde3edcce1e08ec922bc7607f18f717', 'parent': ['5d8e8343c56cf3184081f71cc4a661af64e7cf7e'], 'author': 'CYH <1131637423@qq.com>', 'author_time': 1617625010, 'author_tz': '+0800', 'committer': 'CYH <1131637423@qq.com>', 'committer_time': 1617625010, 'committer_tz': '+0800', 'message': 'Add data parallelism support\n'}


In [18]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [19]:
# Above cell only handles 1 project. Fixed Code Here
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].copy()
dfinf.rename(columns={'author_time': 'time'}, inplace=True)
print(dfinf['project'].nunique())
print(dfinf['project'].unique())

10
['intel-isl_multiobjectiveoptimization' 'sjmoran_deeplpf'
 'pangeo-forge_pangeo-forge' 'lvrcek_gnnome-assembly'
 'josiahcarlson_redis-in-action' 'nmoehrle_mvs-texturing'
 'climateglobalchange_tempestremap' 'miguelinux314_experiment-notebook'
 'rllab-snu_unsupervised-3d-reconstruction' 'coryshain_cdr']


In [20]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,intel-isl_multiobjectiveoptimization,034c7061f35f862db0220636e9a671efd3ba806c,CYH <1131637423@qq.com>,1617625010,Add data parallelism support\n


In [21]:
#mode a means append, so you have all your projects in the same file
yournetid='astubbin'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [22]:
file_path = "astubbin_project_summary.csv"
df = pd.read_csv(file_path, sep=";", names=["project", "commit", "author", "time", "message"],)

grouped = (
    df.groupby("project")
    .agg(
        num_commits=("commit", "count"),
        num_authors=("author", "nunique"),
        first_commit_date=("time", "min"),
        last_commit_date=("time", "max"),
    )
    .reset_index()
)

grouped.columns = [
    "Project",
    "Number of Commits",
    "Number of Authors",
    "Min Time",
    "Max Time",
]

print(grouped.to_csv(sep=";", index=False))

Project;Number of Commits;Number of Authors;Min Time;Max Time
climateglobalchange_tempestremap;718;28;1394210419;1771079955
coryshain_cdr;12339;252;1358361552;1771079955
intel-isl_multiobjectiveoptimization;13;7;1548778261;1725305201
josiahcarlson_redis-in-action;306;52;1358361552;1739979030
lvrcek_gnnome-assembly;377;6;1607052950;1731847672
miguelinux314_experiment-notebook;4165;23;1585572023;1759541950
nmoehrle_mvs-texturing;331;43;1417558133;1755588760
pangeo-forge_pangeo-forge;3105;73;1595348029;1762191998
rllab-snu_unsupervised-3d-reconstruction;434;6;1548679562;1729731600
sjmoran_deeplpf;348;5;1582549772;1706624565



```csv
Project;Number of Stars;Number of Forks;Last Commit Date
intel-isl_multiobjectiveoptimization;1100;173;Sep 2, 2024
sjmoran_deeplpf;258;30;Sep 16, 2026
pangeo-forge_pangeo-forge;134;52;Oct 20,2025
lvrcek_gnnome-assembly;72;10;Nov 17, 2024
josiahcarlson_redis-in-action;2300;1200;Apr 16, 2026
nmoehrle_mvs-texturing;1100;370;Mar 11, 2026
climateglobalchange_tempestremap;53;30;Aug 30, 2026
miguelinux314_experiment-notebook;8;8;Nov 12, 2025
rllab-snu_unsupervised-3d-reconstruction;46;4;Oct 27, 2025
coryshain_cdr;42;11;Jan 22, 2025
```

```csv
Project;Number of Stars;Number of Forks;Last Commit Date;Number of Commits;Number of Authors;Min Time;Max Time
intel-isl_multiobjectiveoptimization;1100;173;Sep 2, 2024;13;7;1548778261;1725305201
sjmoran_deeplpf;258;30;Sep 16, 2026;348;5;1582549772;1706624565
pangeo-forge_pangeo-forge;134;52;Oct 20,2025;3105;73;1595348029;1762191998
lvrcek_gnnome-assembly;72;10;Nov 17, 2024;377;6;1607052950;1731847672
josiahcarlson_redis-in-action;2300;1200;Apr 16, 2026;306;52;1358361552;1739979030
nmoehrle_mvs-texturing;1100;370;Mar 11, 2026;331;43;1417558133;1755588760
climateglobalchange_tempestremap;53;30;Aug 30, 2026;718;28;1394210419;1771079955
miguelinux314_experiment-notebook;8;8;Nov 12, 2025;4165;23;1585572023;1759541950
rllab-snu_unsupervised-3d-reconstruction;46;4;Oct 27, 2025;434;6;1548679562;1729731600
coryshain_cdr;42;11;Jan 22, 2025;12339;252;1358361552;1771079955
```

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github